[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/spin_transformations/spin_transformations.ipynb)

# Spin Transformations

A spin transformation bends a surface while keeping every angle: a quaternion at each vertex turns and scales the edges around it. [Crane, Pinkall and Schröder (2011)](https://www.cs.cmu.edu/~kmcrane/Projects/SpinTransformations/paper.pdf) choose the quaternions so that the surface's mean curvature changes by a prescribed amount. In Cl(3,0) the quaternions are the even subalgebra and the edges are vectors. Every operator below is a sparse extensor: a linear map between fields on the vertices, edges and faces of a mesh, in which each element couples to only a few others, each coupling an extensor.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from examples.geometry.spin_transformations import render
from examples.geometry.spin_transformations.core import (
    Even, Vector, as_diag, as_ga_sparse, as_scalar, context, conformal_smooth, cube, dirac_spheres,
    icosphere, spin_transform_deform, spin_transform_deform_maps,
)

np.set_printoptions(precision=4, suppress=True)
mv = context.multivector

mesh = icosphere(3)                                                     # a unit sphere
print(mesh.vertices.shape, mesh.edges.shape, mesh.faces.shape)
render.draw_surface(mesh, mesh.triangle_areas * 0.0, 1.0, "the sphere");

## 1. Operators between fields

A field is an extensor with one element per vertex, edge or face, such as `mesh.vertices`, `[V] Vector`. An operator between fields is a sparse extensor, a linear map that couples each element of one field to a few elements of another, each coupling an extensor. Here the couplings are multivectors, nullary extensors, which act by their product: times a field, each coupling multiplies its input element into its output element, and the products are summed; times another operator it composes the two; and `~` is its reverse, with inputs and outputs swapped and every coupling reversed. The mesh's incidences say which elements couple. These operators couple through scalars.

In [ ]:
# the face-vertex, face-edge and edge-vertex incidences, and the relative orientations
I20, I21, I10 = mesh.faces, mesh.face_edges, mesh.edges                 # [F, 3], [F, 3], [E, 2]
O10 = np.ones_like(I10) * [-1, 1]                                       # [E, 2]
O21 = mesh.face_edge_orientation                                        # [F, 3]

# the boundary, taking each edge's tail from its head, and the means over each edge and each face
T10 = as_ga_sparse(I10, as_scalar(O10))                                 # [E, V] Scalar
A10 = as_ga_sparse(I10, as_scalar(np.ones_like(I10) / 2))               # [E, V] Scalar
A20 = as_ga_sparse(I20, as_scalar(np.ones_like(I20) / 3))               # [F, V] Scalar

# diagonal operators: the triangle areas, the vertex areas, and each edge's cotangent weight
M2 = as_diag(mesh.triangle_areas)                                       # [F, F] Scalar
M2i = as_diag(1 / mesh.triangle_areas)                                  # [F, F] Scalar
M0 = as_diag(mesh.vertex_areas)                                         # [V, V] Scalar
H1 = as_diag(mesh.edge_ratio)                                           # [E, E] Scalar

edges = T10 * mesh.vertices                                             # [E] Vector

print(T10.gatype, T10.shape)
print(edges.gatype, edges.shape)

## 2. A change in mean curvature

The transformation is asked for by a change `rho` in each face's mean curvature, a scalar field on the faces. Here it grows along y: the surface is to bend more on one side and less on the other.

In [ ]:
centroids = A20 * mesh.vertices                                         # [F] Vector
rho = (centroids | mv.y) * 0.5                                          # [F] Scalar

render.draw_surface(mesh, rho, 0.5, "change in mean curvature");

## 3. The Dirac operator

The Dirac operator sends a quaternion field on the vertices to a field on the faces: each face takes the quaternion at each corner by the edge that corner faces, over minus twice the face's area, so the cells of `D` are vectors. `R` puts `rho` times the pseudoscalar on each face's mean, and `A = D - R`, with odd cells, has the transformation as its null field. The energy of a field is the area-weighted squared size of what `A` makes of it, `~A * M2 * A`: odd cells times their reverses, so its cells come out even, quaternions.

In the quaternionic matrix notation of Crane, Pinkall and Schröder this reads as $D_{ij} = -\tfrac{1}{2\mathcal{A}_i} e_j$, $A = D - R$, and the energy $A^{H} M_F A$.

In [ ]:
L = ~T10 * H1 * T10                                                     # [V, V] Scalar

# each face takes the vertex opposite each edge through that edge, oriented counter-clockwise
D = M2i * as_ga_sparse(I20, edges[I21] * O21) * -0.5                    # [F, V] Vector
R = as_diag(rho.dual()) * A20                                           # [F, V] Pseudoscalar
A = D - R                                                               # [F, V] Odd
Q = ~A * M2 * A                                                         # [V, V] Even

print(A.gatype, A.shape)
print(Q.gatype, Q.shape)

## 4. The quaternion field

The quaternions are the field of least energy per unit of vertex area, the least eigenpair of `Q` against the vertex areas. An eigensolve has to know what a linear map acts on, so `Q * Even` and `M0 * Even` turn its multivectors into maps on quaternions. The field is fixed only up to a constant factor, which would turn and scale the whole surface, so it is divided by its area-weighted mean. The colours show how much each face's mean quaternion scales it.

In [ ]:
_, modes = (Q * Even).eigh(M0 * Even, 1)                                # [1, V] Even
mean = (M0 * modes[0]).sum(axis=0) / mesh.vertex_areas.sum(axis=0)      # [] Even
q = modes[0] / mean                                                     # [V] Even
scaling = (A20 * q).scalar_norm_squared() - 1                           # [F] Scalar

render.draw_surface(mesh, scaling, np.abs(scaling.to_array()).max(), "scaling of each face");

## 5. The new surface

Each edge is turned and scaled by the mean quaternion of its ends, by the reverse sandwich `q << e`. The new vertices are those whose edges match the turned ones best, weighted by the cotangent Laplacian: the least-squares solution of `L` acting on vectors, `L * Vector`, which leaves the surface centred. The checkerboard is the sphere's: a conformal map keeps its squares square.

In [ ]:
transformed_edges = (A10 * q) << edges                                  # [E] Vector
b = ~T10 * H1 * transformed_edges                                       # [V] Vector
deformed = mesh.copy(vertices=(L * Vector).lstsq(b))

render.draw_textured(deformed, mesh, "spin transformation");

## 6. Conformal curvature flow

Repeating the transformation, each time asking for a share of every face's departure from the mean curvature to be taken away, rounds a surface step by step while every step keeps its angles. On a cube the edges and faces round, and the corners stay cone points: a conformal map keeps the angle missing there.

In [ ]:
frames = list(conformal_smooth(cube(8), 30, 0.25))                      # [31] Mesh

for frame, title in zip((frames[0], frames[5], frames[-1]), ("the cube", "rounding", "rounded")):
    render.draw_textured(frame, frames[0], title);

## 7. Dirac spheres

With `rho` a constant `n`, the fields of zero energy are the eigenfields of the Dirac operator itself. Their eigenvalues are the integers except −1, each with multiplicity one more than itself, and spinning the sphere by one gives a Dirac sphere. `dirac_spheres` runs the operators above with that constant and spins the sphere by each of the fields of least energy. The higher states are finer, so the sphere is split once more.

Crane, Pinkall and Schröder, §6.2:

> Eigenfunctions of D on the unit sphere are called the spinor spherical harmonics, which can be used to compute fundamental solutions to the Dirac equation for the configuration of a spin-1/2 particle such as an electron in a spherically symmetric potential.

In [ ]:
fine = icosphere(4)
spun = {n: dirac_spheres(fine, n, 3) for n in (1, 2, 3, -2, -3)}       # eigenvalue: ([3] Scalar, [3] Mesh)
energies = np.stack([energy.to_array().ravel() for energy, _ in spun.values()])   # [eigenvalues, 3]

print(energies)
render.draw_dirac({n: spheres for n, (_, spheres) in spun.items()}, fine);

In [ ]:
# checks
# The operators laid out here are the core's; its version with sparse maps agrees.
np.testing.assert_allclose(deformed.vertices.kernel, spin_transform_deform(mesh, rho).vertices.kernel, atol=1e-11)
np.testing.assert_allclose(deformed.vertices.kernel, spin_transform_deform_maps(mesh, rho).vertices.kernel, atol=1e-11)
# Every corner keeps its angle, up to the mesh's resolution.
np.testing.assert_allclose((deformed.corner_cosines() - mesh.corner_cosines()).to_array(), 0.0, atol=0.05)
# The Dirac spheres' fields have zero energy, up to the mesh's resolution against the spectrum's unit spacing.
np.testing.assert_allclose(energies, 0.0, atol=0.05)